# 텍스트 특징 추출(TF-IDF)

### 1. 샘플 코퍼스로 연습

In [ ]:
sample_corpus = [
    '자연어처리 강의를 시작하겠습니다.',
    '자연어처리는 재미있습니다.',
    '밥을 먹고 강의를 듣고 있습니다.',
    '이번 자연어처리 강의는 한국어 자연어처리입니다.'
]

In [ ]:
import sklearn 

In [ ]:
from konlpy.tag import Okt
def my_tokenizer(text):
    return Okt().nouns(text)

In [ ]:
# TfidfVectorizer 객체 생성
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(tokenizer=my_tokenizer)
# from sklearn.feature_extraction.text import CountVectorizer
# vectorizer = CountVectorizer(tokenizer=my_tokenizer)
# 특징 집합과 관련 데이터 모델을 생성
vectorizer.fit(sample_corpus)
print(vectorizer.get_feature_names_out())
# 특징 백터 생성 
sample_dtm = vectorizer.transform(sample_corpus)
print(sample_dtm.toarray())


### 2. 다음 영화 리뷰로 연습

In [1]:
# 1. csv 파일 불러오기
import pandas as pd

df = pd.read_csv('./data/daum_movie_review.csv')
# 결측치 잘못 포커싱함 review를 지정해야 됬는데 review_corpus로 지정하여 재수정함
df = df.dropna(subset=['review'])

# 2. review 컬럼의 데이터를 리스트로 변환 (이때 변수명이 review_corpus가 됩니다)
review_corpus = df['review'].tolist()

# 확인용 출력
print(f"총 리뷰 개수: {len(review_corpus)}")
print(df.head())

총 리뷰 개수: 14725
                                              review  rating        date  \
0                             돈 들인건 티가 나지만 보는 내내 하품만       1  2018.10.29   
1       몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.      10  2018.10.26   
2  이전 작품에 비해 더 화려하고 스케일도 커졌지만.... 전국 맛집의 음식들을 한데 ...       8  2018.10.24   
3                                이 정도면 볼만하다고 할 수 있음!       8  2018.10.22   
4                                               재미있다      10  2018.10.20   

    title  
0  인피니티 워  
1  인피니티 워  
2  인피니티 워  
3  인피니티 워  
4  인피니티 워  


In [2]:
# tokenizer with Okt
from konlpy.tag import Okt
okt = Okt()

stopWords = ['가가', '가기', '각', '가라']
def my_tokenizer(text):
    # 명사 형용사만 빼오기
    selected_words = []

    if not text:
        return []
    
    for word, tag in okt.pos(text, stem=True):
        if tag in ['Noun', 'Adjective']:
            if word not in stopWords and len(word) > 1 :
                selected_words.append(word)
    return selected_words

In [4]:
# TfidfVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(
    tokenizer=my_tokenizer,
    token_pattern=None,

    min_df=3,
    max_df=0.9,
    max_features=5000

)

# 학습과 변환을 동시 진행 **권장하지않음** 검증데이터를 보여주고 학습을 시키면 무의미함
# tfidf_matrix = vectorizer.fit_transform(review_corpus)
vectorizer.fit(review_corpus)
tfidf_matrix = vectorizer.transform(review_corpus)
# 다음부터는 위의 fit_transform 대신 위의 두라인처럼 학습 > 검증 순으로 작업하기


print(tfidf_matrix.shape)
print(vectorizer.get_feature_names_out()[:50])

(14725, 3833)
['가게' '가까이' '가깝다' '가끔' '가난' '가난하다' '가능' '가능성' '가능하다' '가도' '가득' '가득하다'
 '가디언즈' '가루' '가리봉' '가리봉동' '가망' '가면' '가모라' '가미' '가볍다' '가사' '가상하다' '가세' '가수'
 '가슴' '가슴속' '가시' '가안' '가야' '가오' '가요' '가운데' '가위' '가을' '가장' '가정' '가족' '가족사'
 '가족영화' '가즈' '가지' '가짜' '가치' '가해자' '각각' '각박하다' '각본' '각색' '각성']
